**Task1. Design the Schema Recommendation Logic**


The SchemaRecommender should take analyzed source fields from multiple client datasets and recommend a unified target schema for onboarding. The goal is not only to rename fields, but to decide how common business entities should be represented in a standardized structure.

Requirements:
1. Group source fields that represent the same business meaning.
2. Recommend target tables and target columns.
3. For each target column, provide:
   - field_name
   - data_type
   - nullable
   - description
   - source_coverage
   - design_reason
4. Mark whether each field is required or optional.
5. Explain ambiguous design choices, such as:
   - whether names should be stored as full_name or split into first_name / last_name
   - whether geography should be represented as city and country
   - whether monetary values should keep original currency and also include normalized USD
6. If some clients do not contain a target field, note that clearly.
7. Return the result in structured JSON.

Unified_customers Schema
| field_name          | data_type | nullable | description                                                | source_coverage                 | design_reason                                                            |
| ------------------- | --------- | -------: | ---------------------------------------------------------- | ------------------------------- | ------------------------------------------------------------------------ |
| `customer_id`       | string    |       No | Unified customer identifier used across all source systems | A, B, C                         | Core primary key for customer-level integration                          |
| `first_name`        | string    |      Yes | Customer first name                                        | Mainly C, optional for A/B      | Client C already separates names; useful for standardized downstream use |
| `last_name`         | string    |      Yes | Customer last name                                         | Mainly C, optional for A/B      | Client C already separates names; useful for standardized downstream use |
| `full_name`         | string    |      Yes | Customer full name in a single field                       | A, B, derivable for C           | Keeps compatibility with sources where name is not split                 |
| `email`             | string    |      Yes | Customer email address                                     | A, B, C                         | Common contact field across clients                                      |
| `phone`             | string    |      Yes | Customer phone number                                      | A, B, C                         | Common contact field, but often missing or inconsistently formatted      |
| `registration_date` | date      |      Yes | Date when the customer registered or signed up             | A, B, C                         | Shared business meaning across clients despite different date formats    |
| `city`              | string    |      Yes | Customer city                                              | B primarily, optional elsewhere | Useful geography field; may not exist in all sources                     |
| `country`           | string    |      Yes | Customer country or country code                           | C primarily, optional elsewhere | Useful geography field; may be missing in some sources                   |
| `source_system`     | string    |       No | Name of the source client/system that the record came from | A, B, C                         | Required for lineage, traceability, and debugging                        |


Unified_orders Schema
| field_name         | data_type | nullable | description                                                | source_coverage                    | design_reason                                                                                           |
| ------------------ | --------- | -------: | ---------------------------------------------------------- | ---------------------------------- | ------------------------------------------------------------------------------------------------------- |
| `order_id`         | string    |       No | Unified order identifier                                   | A, B, C                            | Core primary key for order-level integration                                                            |
| `customer_id`      | string    |       No | Unified customer identifier linked to the order            | A, B, C                            | Required foreign key connecting orders to customers                                                     |
| `order_date`       | date      |      Yes | Date when the order was placed                             | A, B, C                            | Shared transaction date field across clients despite format differences and occasional malformed values |
| `amount`           | decimal   |       No | Original order amount in source currency                   | A, B, C                            | Core transaction value that should preserve original monetary amount                                    |
| `currency`         | string    |      Yes | Currency code of the original order amount                 | B, C explicitly; defaultable for A | Needed to preserve monetary context and support normalization                                           |
| `amount_usd`       | decimal   |      Yes | Order amount normalized to USD                             | Derived for A, B, C                | Useful standardized monetary field for cross-client comparison                                          |
| `order_status`     | string    |      Yes | Standardized order lifecycle status                        | A, B, C                            | Important business state field; supports translation across languages                                   |
| `shipping_address` | string    |      Yes | Shipping or delivery address for the order                 | A, B, C                            | Common order fulfillment field, though sometimes missing                                                |
| `source_system`    | string    |       No | Name of the source client/system that the record came from | A, B, C                            | Required for lineage, traceability, and debugging                                                       |


**Task5 issue ands fix log**


Client a:
- Issue 1:
  The generated mapping left `first_name` and `last_name` as null.

- Fix:
  Updated the prompt to require splitting `full_name` into `first_name` and `last_name` for client_a.

- Issue 2:
  The generated mapping set `currency` to null and made `amount_usd` conditional.

- Fix:
  Updated the prompt to default `currency` to USD for client_a and map `amount_usd` directly from `total_amt`.


Client b:
- Issue:
  The generated mapping left `first_name` and `last_name` as null, and translated `terkirim` to `shipped`.
- Why it is a problem:
  The current project design requires splitting `nama` into structured name fields when possible, and the unified status vocabulary should use `delivered`, `pending`, and `cancelled`.
- Fix applied:
  Refined the mapping prompt to require splitting `nama` into `first_name` and `last_name`, and to map `terkirim` to `delivered`.


Client c:

- Issue:
  The generated mapping kept `Telefon` too close to the raw source format, used `canceled` instead of `cancelled`, and did not make formatting cleanup for `Land` and `Währung` explicit.

- Why it is a problem:
  The current project design prefers normalizing phone values into a standard format, keeping `country` and `currency` codes clean and uppercase, and using the unified status vocabulary `delivered`, `pending`, and `cancelled`.

- Fix applied:
  Refined the mapping prompt to require `format_cleanup` for `Telefon`, normalization of `Land` and `Währung` with trimming and uppercase formatting, and translation of German statuses so that `Storniert` maps to `cancelled` instead of `canceled`.